<a href="https://colab.research.google.com/github/nlknnaailkhan/GoodnessFFSentimentAnalysis/blob/main/forwardforward.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install omegaconf gensim

import os
import time
import math
import random
import kagglehub
from datetime import timedelta
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset
from omegaconf import OmegaConf

from gensim.models import Word2Vec
import gensim
import nltk
import warnings
from nltk.tokenize import sent_tokenize, word_tokenize
import pandas as pd
import gensim.downloader as api

print("Loading Word2Vec embeddings (this may take a minute)...")
word2vec = api.load("word2vec-google-news-300")
print("Embeddings loaded successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 34.7 MB/s eta 0:00:00
Loading Word2Vec embeddings (this may take a minute)...
[==================================================] 100.0% 1662.8/1662.8MB downloaded
Embeddings loaded successfully!


In [ ]:
config_dict = {
    "seed": 100,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "input": {
        "path": "datasets",
        "batch_size": 100,
        "num_classes": 10,
        "num_features": 310
    },
    "model": {
        "peer_normalization": 0.03,
        "momentum": 0.9,
        "hidden_dim": 200,
        "num_layers": 3  ,
        "goodness_type": "log_sum_exp"
    },    "training": {
        "epochs": 50,
        "learning_rate": 3e-4,
        "weight_decay": 3e-4,
        "momentum": 0.9,
        "downstream_learning_rate": 1e-2,
        "downstream_weight_decay": 3e-3,
        "val_idx": -1,
        "final_test": True

    }
}

opt = OmegaConf.create(config_dict)

print(f"PyTorch CUDA Available: {torch.cuda.is_available()}")
print(f"Configured Device: {opt.device}")

PyTorch CUDA Available: True
Configured Device: cuda


In [ ]:
goodness_functions={
    "sum_of_squares":  lambda h: torch.sum(h ** 2, dim=-1),
    "log_sum_exp":      lambda h: torch.logsumexp(h, dim=-1),
    "sparse_l1":       lambda h: torch.sum(h ** 2, dim=-1) - 0.1 * torch.sum(torch.abs(h), dim=-1),
    "huber_norm":      lambda h: torch.sum(torch.where(torch.abs(h) <= 20.0, 0.5 * (h ** 2), 1.0 * (torch.abs(h) - 0.5)), dim=-1),
    "tempered_energy": lambda h: torch.sum(torch.exp((h ** 2) / 20.0), dim=-1),
    "outlier_trimmed": lambda h: torch.sum(torch.topk(h ** 2, k=int(h.shape[-1] * 0.9), dim=-1, largest=False).values, dim=-1),
    "ojas_rule":       lambda h: torch.sum(h ** 2, dim=-1) - 0.1 * torch.sum(h ** 4, dim=-1),


}

In [ ]:
def parse_args(opt):
    np.random.seed(opt.seed)
    torch.manual_seed(opt.seed)
    random.seed(opt.seed)
    print(OmegaConf.to_yaml(opt))
    return opt

def get_model_and_optimizer(opt):
    model = FF_model(opt)
    if "cuda" in opt.device:
        model = model.cuda()
    print(model, "\n")

    main_model_params = [
        p for p in model.parameters()
        if all(p is not x for x in model.linear_classifier.parameters())
    ]
    optimizer = torch.optim.SGD([
        {
            "params": main_model_params,
            "lr": opt.training.learning_rate,
            "weight_decay": opt.training.weight_decay,
            "momentum": opt.training.momentum,
        },
        {
            "params": model.linear_classifier.parameters(),
            "lr": opt.training.downstream_learning_rate,
            "weight_decay": opt.training.downstream_weight_decay,
            "momentum": opt.training.momentum,
        },
    ])
    return model, optimizer

def get_data(opt, partition):
    images_path = os.path.join(os.getcwd(), opt.input.path, f"{partition}_images.npy")
    labels_path = os.path.join(os.getcwd(), opt.input.path, f"{partition}_labels.npy")

    dataset = FF_Data(images_path=images_path, labels_path=labels_path, num_classes=opt.input.num_classes)

    if partition == "train":
        permutation = np.random.permutation(len(dataset.images))
        dataset.images = dataset.images[permutation]
        dataset.labels = dataset.labels[permutation]

    g = torch.Generator()
    g.manual_seed(opt.seed)

    return torch.utils.data.DataLoader(
        dataset,
        batch_size=opt.input.batch_size,
        drop_last=True,
        shuffle=True,
        worker_init_fn=seed_worker,
        generator=g,
        num_workers=2,
        persistent_workers=True,
    )

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2 ** 32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def dict_to_cuda(d):
    for key, value in d.items():
        d[key] = value.cuda(non_blocking=True)
    return d

def preprocess_inputs(opt, inputs, labels):
    if "cuda" in opt.device:
        inputs = dict_to_cuda(inputs)
        labels = dict_to_cuda(labels)
    return inputs, labels

def get_linear_cooldown_lr(opt, epoch, lr):
    if epoch > (opt.training.epochs // 2):
        return lr * 2 * (1 + opt.training.epochs - epoch) / opt.training.epochs
    else:
        return lr

def update_learning_rate(optimizer, opt, epoch):
    optimizer.param_groups[0]["lr"] = get_linear_cooldown_lr(
        opt, epoch, opt.training.learning_rate
    )
    optimizer.param_groups[1]["lr"] = get_linear_cooldown_lr(
        opt, epoch, opt.training.downstream_learning_rate
    )
    return optimizer

def get_accuracy(opt, output, target):
    with torch.no_grad():
        prediction = torch.argmax(output, dim=1)
        return (prediction == target).sum() / opt.input.batch_size

def print_results(partition, iteration_time, scalar_outputs, epoch=None):
    if epoch is not None:
        print(f"Epoch {epoch} \t", end="")
    print(f"{partition} \t \tTime: {timedelta(seconds=iteration_time)} \t", end="")
    if scalar_outputs is not None:
        for key, value in scalar_outputs.items():
            print(f"{key}: {value:.4f} \t", end="")
    print()

def log_results(result_dict, scalar_outputs, num_steps):
    for key, value in scalar_outputs.items():
        if isinstance(value, float):
            result_dict[key] += value / num_steps
        else:
            result_dict[key] += value.item() / num_steps
    return result_dict

In [ ]:
print("Downloading Rotten Tomatoes dataset...")
path = kagglehub.dataset_download("mrbaloglu/rotten-tomatoes-reviews-dataset")
print("Path to dataset files:", path)

output_dir = "datasets"
os.makedirs(output_dir, exist_ok=True)

csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
csv_path = os.path.join(path, csv_file)
df = pd.read_csv(csv_path)

raw_labels = df.iloc[:, 0].values
raw_reviews = df.iloc[:, 1].values

np.random.seed(42)  # For consistent split behavior
shuffle_indices = np.random.permutation(len(raw_reviews))
shuffled_reviews = raw_reviews[shuffle_indices]
shuffled_labels = raw_labels[shuffle_indices]

print("Vectorizing reviews...")
embedded_features = []
for sentence in shuffled_reviews:
    words = str(sentence).lower().split()
    vectors = [word2vec[w] for w in words if w in word2vec]

    if len(vectors) > 0:
        mean_vector = np.mean(vectors, axis=0)
    else:
        mean_vector = np.zeros(300)

    embedded_features.append(mean_vector)

embedded_features = np.array(embedded_features, dtype=np.float32)

total_samples = len(embedded_features)
train_end = int(total_samples * 0.8)
val_end = int(total_samples * 0.9)

train_images = embedded_features[:train_end]
train_labels = shuffled_labels[:train_end]

val_images = embedded_features[train_end:val_end]
val_labels = shuffled_labels[train_end:val_end]

test_images = embedded_features[val_end:]
test_labels = shuffled_labels[val_end:]

np.save(os.path.join(output_dir, "train_images.npy"), train_images)
np.save(os.path.join(output_dir, "train_labels.npy"), train_labels.astype(np.int64))

np.save(os.path.join(output_dir, "val_images.npy"), val_images)
np.save(os.path.join(output_dir, "val_labels.npy"), val_labels.astype(np.int64))

# ADDED: Overwrite the old FashionMNIST test arrays with text vectors
np.save(os.path.join(output_dir, "test_images.npy"), test_images)
np.save(os.path.join(output_dir, "test_labels.npy"), test_labels.astype(np.int64))

print(f"\nSuccess! Transformed total text dataset into clean files.")

Using Colab cache for faster access to the 'rotten-tomatoes-reviews-dataset' dataset.
Path to dataset files: /kaggle/input/rotten-tomatoes-reviews-dataset
Vectorizing reviews...

Success! Transformed total text dataset into clean files.


In [ ]:
class FF_Data(Dataset):
    def __init__(self, images_path, labels_path, num_classes=10):
        self.images = np.load(images_path)
        self.labels = np.load(labels_path)
        self.num_classes = num_classes
        self.uniform_label = torch.ones(self.num_classes) / self.num_classes

    def __getitem__(self, index):
        pos_sample, neg_sample, neutral_sample, class_label = self._generate_sample(index)

        inputs = {
            "pos_images": pos_sample,
            "neg_images": neg_sample,
            "neutral_sample": neutral_sample,
        }
        labels = {"class_labels": class_label}
        return inputs, labels

    def __len__(self):
        return len(self.images)

    def _get_pos_sample(self, sample, class_label):
        one_hot_label = torch.nn.functional.one_hot(
            torch.tensor(class_label), num_classes=self.num_classes
        ).float()


        return torch.cat([sample, one_hot_label], dim=-1)

    def _get_neg_sample(self, sample, class_label):
        classes = list(range(self.num_classes))
        classes.remove(class_label)
        wrong_class_label = np.random.choice(classes)
        one_hot_label = torch.nn.functional.one_hot(
            torch.tensor(wrong_class_label), num_classes=self.num_classes
        ).float()

        return torch.cat([sample, one_hot_label], dim=-1)

    def _get_neutral_sample(self, sample):
        return torch.cat([sample, self.uniform_label], dim=-1)

    def _generate_sample(self, index):
        sample = torch.tensor(self.images[index], dtype=torch.float32)
        if sample.max() > 1.0:
            sample = sample / 255.0
        class_label = int(self.labels[index])

        sample = sample.flatten()

        pos_sample = self._get_pos_sample(sample, class_label)
        neg_sample = self._get_neg_sample(sample, class_label)

        neutral_sample = self._get_neutral_sample(sample)

        return pos_sample, neg_sample, neutral_sample, class_label

In [ ]:
class ReLU_full_grad(torch.autograd.Function):
    """ ReLU activation function that passes through the gradient irrespective of its input value. """
    @staticmethod
    def forward(ctx, input):
        return input.clamp(min=0)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.clone()


class FF_model(torch.nn.Module):
    """The model trained with Forward-Forward (FF)."""
    def __init__(self, opt):
        super(FF_model, self).__init__()

        self.opt = opt
        self.num_channels = [self.opt.model.hidden_dim] * self.opt.model.num_layers

        self.act_fn = ReLU_full_grad

        self.model = nn.ModuleList([nn.Linear(self.opt.input.num_features, self.num_channels[0])])
        for i in range(1, len(self.num_channels)):
            self.model.append(nn.Linear(self.num_channels[i - 1], self.num_channels[i]))

        self.ff_loss = nn.BCEWithLogitsLoss()
        self.threshold = None

        self.running_means = [
            torch.zeros(self.num_channels[i], device=self.opt.device) + 0.5
            for i in range(self.opt.model.num_layers)
        ]

        channels_for_classification_loss = sum(
            self.num_channels[-i] for i in range(self.opt.model.num_layers - 1)
        )
        self.linear_classifier = nn.Sequential(
            nn.Linear(channels_for_classification_loss, 10, bias=False)
        )
        self.classification_loss = nn.CrossEntropyLoss()
        self._init_weights()

    def _init_weights(self):
        for m in self.model.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.normal_(
                    m.weight, mean=0, std=1 / math.sqrt(m.weight.shape[0])
                )
                torch.nn.init.zeros_(m.bias)

        for m in self.linear_classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.zeros_(m.weight)

    def _layer_norm(self, z, eps=1e-8):
        return z / (torch.sqrt(torch.mean(z ** 2, dim=-1, keepdim=True)) + eps)

    def _calc_peer_normalization_loss(self, idx, z):
        mean_activity = torch.mean(z[:self.opt.input.batch_size], dim=0)

        self.running_means[idx] = self.running_means[idx].detach() * self.opt.model.momentum + mean_activity * (1 - self.opt.model.momentum)
        peer_loss = (torch.mean(self.running_means[idx]) - self.running_means[idx]) ** 2
        return torch.mean(peer_loss)

    def _calc_ff_loss(self, z, labels):
        goodness_type = getattr(self.opt.model, 'goodness_type', 'sum_of_squares')
        goodness_fn = goodness_functions.get(goodness_type, goodness_functions['sum_of_squares'])
        goodness = goodness_fn(z)

        if 'mean' in goodness_type:
            default_threshold = 2.0
        else:
            default_threshold = z.shape[1]

        current_threshold = getattr(self, 'threshold', default_threshold)
        logits = goodness - current_threshold

        ff_loss = self.ff_loss(logits, labels.float())

        with torch.no_grad():
          ff_accuracy = (
              torch.sum((torch.sigmoid(logits) > 0.5) == labels) / z.shape[0]
          ).item()

        return ff_loss, ff_accuracy

    def forward(self, inputs, labels):
        scalar_outputs = {
            "Loss": torch.zeros(1, device=self.opt.device),
            "Peer Normalization": torch.zeros(1, device=self.opt.device),
        }

        z = torch.cat([inputs["pos_images"], inputs["neg_images"]], dim=0)
        posneg_labels = torch.zeros(z.shape[0], device=self.opt.device)
        posneg_labels[: self.opt.input.batch_size] = 1

        z = z.reshape(z.shape[0], -1)
        z = self._layer_norm(z)

        for idx, layer in enumerate(self.model):
            z = layer(z)

            z = self.act_fn.apply(z)

            if self.opt.model.peer_normalization > 0:
                peer_loss = self._calc_peer_normalization_loss(idx, z)
                scalar_outputs["Peer Normalization"] += peer_loss
                scalar_outputs["Loss"] += self.opt.model.peer_normalization * peer_loss

            ff_loss, ff_accuracy = self._calc_ff_loss(z, posneg_labels)
            scalar_outputs[f"loss_layer_{idx}"] = ff_loss
            scalar_outputs[f"ff_accuracy_layer_{idx}"] = ff_accuracy
            scalar_outputs["Loss"] += ff_loss
            z = z.detach()
            z = self._layer_norm(z)

        scalar_outputs = self.forward_downstream_classification_model(
            inputs, labels, scalar_outputs=scalar_outputs
        )
        return scalar_outputs

    def forward_downstream_classification_model(self, inputs, labels, scalar_outputs=None):
        if scalar_outputs is None:
            scalar_outputs = {
                "Loss": torch.zeros(1, device=self.opt.device),
            }

        z = inputs["neutral_sample"]
        z = z.reshape(z.shape[0], -1)
        z = self._layer_norm(z)
        input_classification_model = []

        with torch.no_grad():
            for idx, layer in enumerate(self.model):
                z = layer(z)
                z = self.act_fn.apply(z)
                z = self._layer_norm(z)
                if idx >= 1:
                    input_classification_model.append(z)

        input_classification_model = torch.concat(input_classification_model, dim=-1)
        output = self.linear_classifier(input_classification_model.detach())
        output = output - torch.max(output, dim=-1, keepdim=True)[0]
        classification_loss = self.classification_loss(output, labels["class_labels"])
        classification_accuracy = get_accuracy(self.opt, output.data, labels["class_labels"])

        scalar_outputs["Loss"] += classification_loss
        scalar_outputs["classification_loss"] = classification_loss
        scalar_outputs["classification_accuracy"] = classification_accuracy
        return scalar_outputs

In [ ]:
def train(opt, model, optimizer):
    start_time = time.time()
    train_loader = get_data(opt, "train")
    num_steps_per_epoch = len(train_loader)

    for epoch in range(opt.training.epochs):
        model.train()
        train_results = defaultdict(float)
        optimizer = update_learning_rate(optimizer, opt, epoch)

        for inputs, labels in train_loader:
            inputs, labels = preprocess_inputs(opt, inputs, labels)
            optimizer.zero_grad()
            scalar_outputs = model(inputs, labels)
            scalar_outputs["Loss"].backward()
            optimizer.step()
            train_results = log_results(train_results, scalar_outputs, num_steps_per_epoch)

        print_results("train", time.time() - start_time, train_results, epoch)
        start_time = time.time()

        if epoch == 50:
            validate_or_test(opt, model, "val", epoch=epoch)

    return model

def validate_or_test(opt, model, partition, epoch=None):
    test_time = time.time()
    test_results = defaultdict(float)

    data_loader = get_data(opt, partition)
    num_steps_per_epoch = len(data_loader)

    model.eval()
    print(partition)
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = preprocess_inputs(opt, inputs, labels)
            scalar_outputs = model.forward_downstream_classification_model(inputs, labels)
            test_results = log_results(test_results, scalar_outputs, num_steps_per_epoch)

    print_results(partition, time.time() - test_time, test_results, epoch=epoch)
    model.train()


In [ ]:
# --- Main Execution ---
print("Parsing Configurations...")
parsed_opt = parse_args(opt)

print("\nInitializing Model and Optimizer...")
model, optimizer = get_model_and_optimizer(parsed_opt)

init_loader = get_data(parsed_opt, "train")
first_inputs, first_labels = next(iter(init_loader))
first_inputs, first_labels = preprocess_inputs(parsed_opt, first_inputs, first_labels)

z_init = torch.cat([first_inputs["pos_images"], first_inputs["neg_images"]], dim=0)
z_init = z_init.reshape(z_init.shape[0], -1)

with torch.no_grad():
    z_init = model._layer_norm(z_init)
    z_init = model.model[0](z_init)
    z_init = model.act_fn.apply(z_init)

    goodness_type = getattr(opt.model, 'goodness_type', 'sum_of_squares')
    goodness_fn = goodness_functions.get(goodness_type, goodness_functions['sum_of_squares'])
    raw_goodness=goodness_fn(z_init)
    threshold = torch.mean(raw_goodness).item()

model.threshold = threshold

print("\nStarting Training Loop...")
model = train(parsed_opt, model, optimizer)

print("\nRunning Final Validation...")
validate_or_test(parsed_opt, model, "val")

if parsed_opt.training.final_test:
    print("\nRunning Final Test...")
    validate_or_test(parsed_opt, model, "test")

Parsing Configurations...
seed: 100
device: cuda
input:
  path: datasets
  batch_size: 100
  num_classes: 10
  num_features: 310
model:
  peer_normalization: 0.03
  momentum: 0.9
  hidden_dim: 200
  num_layers: 3
  goodness_type: log_sum_exp
training:
  epochs: 50
  learning_rate: 0.0003
  weight_decay: 0.0003
  momentum: 0.9
  downstream_learning_rate: 0.01
  downstream_weight_decay: 0.003
  val_idx: -1
  final_test: true


Initializing Model and Optimizer...
FF_model(
  (model): ModuleList(
    (0): Linear(in_features=310, out_features=200, bias=True)
    (1-2): 2 x Linear(in_features=200, out_features=200, bias=True)
  )
  (ff_loss): BCEWithLogitsLoss()
  (linear_classifier): Sequential(
    (0): Linear(in_features=400, out_features=10, bias=False)
  )
  (classification_loss): CrossEntropyLoss()
) 


Starting Training Loop...
Epoch 0 	train 	 	Time: 0:00:02.460536 	Loss: 2.7803 	Peer Normalization: 0.6293 	loss_layer_0: 0.6990 	ff_accuracy_layer_0: 0.4626 	loss_layer_1: 0.6860 	ff_a